# 04 — Indicateurs climatiques projetés : CMIP6 × ERA5

**Objectif** : construire des indicateurs climatiques projetés en combinant les réanalyses **ERA5** (observations historiques) avec les modèles de projection **CMIP6**, en appliquant des méthodes de **bias correction / downscaling statistique**.

---

## Workflow

```
ERA5 (0.25°, 1979–2023)          CMIP6 GCMs (~1-2°, 1850–2100)
    │ observations historiques         │ projections SSP
    │                                  │
    └──────────┬───────────────────────┘
               │
     Bias Correction (quantile mapping)
               │
    ┌──────────┴──────────────┐
    │  Indicateurs projetés   │
    │  corrigés et downscalés │
    └─────────────────────────┘
```

## Méthodes de downscaling explorées

| Méthode | Complexité | Description |
|---------|------------|-------------|
| **Delta method** | Simple | Applique le changement GCM (futur - hist) sur ERA5 |
| **Quantile mapping** | Moyen | Corrige la distribution CDF du GCM vers ERA5 |
| **BCSD** | Avancé | Bias Correction + Spatial Disaggregation |

## Indicateurs climatiques

| Indicateur | Variable | Seuil | Unité |
|------------|----------|-------|-------|
| Jours de chaleur extrême | tas (T2m) | > 35°C | jours/an |
| Nuits tropicales | tasmin | > 20°C | nuits/an |
| Jours de gel | tasmin | < 0°C | jours/an |
| Précipitations extrêmes | pr | > 20 mm/j | jours/an |
| Jours secs consécutifs (CDD) | pr | < 1 mm/j | jours max |
| Degrés-jours de refroidissement (CDD) | tas | base 22°C | °C·jours/an |

---

**Prérequis** : un compte CDS (Copernicus Climate Data Store) avec clé API configurée dans `~/.cdsapirc`.

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from scipy import stats

# Chemins
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "climate"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Répertoire données climat : {DATA_DIR}")
print("Imports OK")

Répertoire données climat : /Users/alexandre/Documents/Geospatial/open_climate_risk/data/climate
Imports OK


## 1. Configuration — Zone d'étude et paramètres

On définit une zone d'étude (bounding box) et les paramètres des données à télécharger.

In [2]:
# ── Zone d'étude : France métropolitaine ──────────────────────────────────────
BBOX = {
    "north": 51.5,
    "south": 41.0,
    "west": -5.5,
    "east": 10.0,
}

# ── Périodes ──────────────────────────────────────────────────────────────────
HIST_PERIOD = (1991, 2020)    # Baseline climatologique (30 ans)
FUTURE_PERIODS = {
    "near":  (2021, 2050),    # Court terme
    "mid":   (2041, 2070),    # Moyen terme
    "far":   (2071, 2100),    # Long terme
}

# ── Scénarios SSP (CMIP6) ────────────────────────────────────────────────────
# SSP remplacent les RCP de CMIP5 :
# SSP1-2.6 : durabilité (↔ RCP 2.6)
# SSP2-4.5 : milieu de route (↔ RCP 4.5)
# SSP3-7.0 : rivalités régionales
# SSP5-8.5 : développement fossile (↔ RCP 8.5)
SSP_SCENARIOS = ["ssp126", "ssp245", "ssp370", "ssp585"]

SSP_COLORS = {
    "ssp126": "#2196F3",  # bleu
    "ssp245": "#FF9800",  # orange
    "ssp370": "#e63946",  # rouge
    "ssp585": "#9C27B0",  # violet
}

SSP_LABELS = {
    "ssp126": "SSP1-2.6 (durabilité)",
    "ssp245": "SSP2-4.5 (milieu de route)",
    "ssp370": "SSP3-7.0 (rivalités)",
    "ssp585": "SSP5-8.5 (fossile)",
}

# ── Modèles CMIP6 sélectionnés ───────────────────────────────────────────────
# Sous-ensemble robuste, bien représenté sur CDS
CMIP6_MODELS = [
    "ipsl_cm6a_lr",
    "mri_esm2_0",
    "gfdl_esm4",
    "ukesm1_0_ll",
    "mpi_esm1_2_lr",
]

print(f"Zone : {BBOX}")
print(f"Baseline : {HIST_PERIOD}")
print(f"Scénarios : {SSP_SCENARIOS}")
print(f"Modèles : {len(CMIP6_MODELS)}")

Zone : {'north': 51.5, 'south': 41.0, 'west': -5.5, 'east': 10.0}
Baseline : (1991, 2020)
Scénarios : ['ssp126', 'ssp245', 'ssp370', 'ssp585']
Modèles : 5


## 2. Téléchargement des données ERA5 via CDS API

ERA5 est la réanalyse de référence du ECMWF :
- Résolution : 0.25° (~31 km)
- Couverture : 1940–présent
- Variables : température, précipitations, vent, humidité, etc.

On télécharge les **moyennes journalières** de température (T2m) et précipitations sur la France.

> **Note** : le téléchargement nécessite un compte CDS et une clé API dans `~/.cdsapirc`.
> Format du fichier : `url: https://cds.climate.copernicus.eu/api` + `key: <votre-clé>`

In [3]:
import cdsapi

def download_era5_daily(variable, years, bbox, output_path):
    """Télécharge des données ERA5 journalières depuis CDS.
    
    Parameters
    ----------
    variable : str
        Variable CDS (ex: '2m_temperature', 'total_precipitation')
    years : list[str]
        Années à télécharger
    bbox : dict
        Bounding box {north, south, west, east}
    output_path : Path
        Chemin de sortie (.nc)
    """
    if output_path.exists():
        print(f"  Déjà téléchargé : {output_path.name}")
        return
    
    client = cdsapi.Client()
    client.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": variable,
            "year": years,
            "month": [f"{m:02d}" for m in range(1, 13)],
            "day": [f"{d:02d}" for d in range(1, 32)],
            "time": "12:00",  # Valeur à midi UTC (approximation journalière)
            "area": [bbox["north"], bbox["west"], bbox["south"], bbox["east"]],
            "data_format": "netcdf",
        },
        str(output_path),
    )
    print(f"  Téléchargé : {output_path.name}")

# ── Téléchargement ERA5 (baseline 1991–2020) ─────────────────────────────────
# ATTENTION : ce téléchargement peut prendre 10-30 min selon la charge CDS
# Décommenter pour lancer le téléchargement

era5_years = [str(y) for y in range(HIST_PERIOD[0], HIST_PERIOD[1] + 1)]

# download_era5_daily(
#     variable="2m_temperature",
#     years=era5_years,
#     bbox=BBOX,
#     output_path=DATA_DIR / "era5_t2m_france_1991_2020.nc",
# )

# download_era5_daily(
#     variable="total_precipitation",
#     years=era5_years,
#     bbox=BBOX,
#     output_path=DATA_DIR / "era5_tp_france_1991_2020.nc",
# )

print("Fonctions de téléchargement ERA5 définies.")
print("→ Décommenter les appels ci-dessus pour lancer le téléchargement.")

Fonctions de téléchargement ERA5 définies.
→ Décommenter les appels ci-dessus pour lancer le téléchargement.


## 3. Téléchargement des données CMIP6 via CDS

CMIP6 (Coupled Model Intercomparison Project Phase 6) fournit les projections climatiques :
- Résolution variable (~1-2° selon le modèle)
- Scénarios SSP (Shared Socioeconomic Pathways)
- Historique (1850–2014) + projections (2015–2100)

On télécharge la température journalière (`tas`) pour les périodes historique et futures.

In [4]:
def download_cmip6(variable, model, experiment, years, bbox, output_path):
    """Télécharge des données CMIP6 journalières depuis CDS.
    
    Parameters
    ----------
    variable : str
        Variable CMIP6 (ex: 'near_surface_air_temperature')
    model : str
        Nom du modèle CDS (ex: 'ipsl_cm6a_lr')
    experiment : str
        Expérience ('historical', 'ssp126', 'ssp245', etc.)
    years : list[str]
        Années à télécharger
    bbox : dict
        Bounding box {north, south, west, east}
    output_path : Path
        Chemin de sortie (.nc)
    """
    if output_path.exists():
        print(f"  Déjà téléchargé : {output_path.name}")
        return
    
    client = cdsapi.Client()
    client.retrieve(
        "projections-cmip6",
        {
            "temporal_resolution": "daily",
            "experiment": experiment,
            "variable": variable,
            "model": model,
            "year": years,
            "month": [f"{m:02d}" for m in range(1, 13)],
            "day": [f"{d:02d}" for d in range(1, 32)],
            "area": [bbox["north"], bbox["west"], bbox["south"], bbox["east"]],
            "data_format": "netcdf",
        },
        str(output_path),
    )
    print(f"  Téléchargé : {output_path.name}")

/
# ── Téléchargement CMIP6 ─────────────────────────────────────────────────────
# Décommenter pour lancer (un téléchargement par modèle × scénario × période)

# Exemple pour IPSL-CM6A-LR, historique
# cmip6_hist_years = [str(y) for y in range(1991, 2015)]  # CMIP6 hist s'arrête en 2014
# download_cmip6(
#     variable="near_surface_air_temperature",
#     model="ipsl_cm6a_lr",
#     experiment="historical",
#     years=cmip6_hist_years,
#     bbox=BBOX,
#     output_path=DATA_DIR / "cmip6_tas_ipsl_historical_france.nc",
# )

# Exemple pour IPSL-CM6A-LR, SSP2-4.5
# cmip6_fut_years = [str(y) for y in range(2015, 2101)]
# download_cmip6(
#     variable="near_surface_air_temperature",
#     model="ipsl_cm6a_lr",
#     experiment="ssp245",
#     years=cmip6_fut_years,
#     bbox=BBOX,
#     output_path=DATA_DIR / "cmip6_tas_ipsl_ssp245_france.nc",
# )

print("Fonctions de téléchargement CMIP6 définies.")
print("→ Décommenter pour lancer les téléchargements.")

Fonctions de téléchargement CMIP6 définies.
→ Décommenter pour lancer les téléchargements.


## 4. Données synthétiques pour démonstration

En attendant les téléchargements CDS, on génère des données synthétiques réalistes pour démontrer le pipeline complet de bias correction et calcul d'indicateurs.

In [ ]:
def generate_synthetic_climate(
    lat_range=(41.0, 51.5),
    lon_range=(-5.5, 10.0),
    res=1.0,
    years_hist=(1991, 2020),
    years_fut=(2041, 2070),
    warming_delta=2.5,
    seed=42,
):
    """Génère des données climatiques synthétiques réalistes pour la France.
    
    Simule :
    - ERA5 (obs) : cycle saisonnier + bruit + gradient latitudinal
    - CMIP6 hist : même structure + biais systématique
    - CMIP6 futur : hist + signal de réchauffement
    
    Returns
    -------
    era5, cmip6_hist, cmip6_fut : xr.DataArray
        Température journalière en °C
    """
    rng = np.random.default_rng(seed)
    
    lats = np.arange(lat_range[0], lat_range[1], res)
    lons = np.arange(lon_range[0], lon_range[1], res)
    
    def _make_series(years, base_temp=12.0, amplitude=12.0, bias=0.0, trend=0.0):
        dates = pd.date_range(f"{years[0]}-01-01", f"{years[1]}-12-31", freq="D")
        n_days = len(dates)
        n_lat, n_lon = len(lats), len(lons)
        
        # Cycle saisonnier
        doy = dates.dayofyear.values
        seasonal = amplitude * np.sin(2 * np.pi * (doy - 100) / 365.25)
        
        # Gradient latitudinal : ~0.6°C par degré de latitude
        lat_effect = -0.6 * (lats[:, None] - 46.0) * np.ones((1, n_lon))
        
        # Tendance linéaire (°C/an)
        year_frac = (dates - dates[0]).days / 365.25
        trend_signal = trend * year_frac
        
        # Bruit journalier
        noise = rng.normal(0, 3.0, size=(n_days, n_lat, n_lon))
        
        # Assemblage
        temp = (
            base_temp + bias
            + seasonal[:, None, None]
            + lat_effect[None, :, :]
            + trend_signal[:, None, None]
            + noise
        )
        
        return xr.DataArray(
            temp,
            dims=["time", "latitude", "longitude"],
            coords={"time": dates, "latitude": lats, "longitude": lons},
            attrs={"units": "°C", "long_name": "Near-surface air temperature"},
        )
    
    # ERA5 : "vérité terrain"
    era5 = _make_series(years_hist, base_temp=12.0, amplitude=12.0, bias=0.0, trend=0.02)
    
    # CMIP6 historique : biais chaud de +1.5°C, amplitude légèrement différente
    cmip6_hist = _make_series(years_hist, base_temp=12.0, amplitude=11.0, bias=1.5, trend=0.02)
    
    # CMIP6 futur : réchauffement additionnel
    cmip6_fut = _make_series(years_fut, base_temp=12.0, amplitude=11.0,
                              bias=1.5 + warming_delta, trend=0.03)
    
    return era5, cmip6_hist, cmip6_fut

# Générer les données
era5_synth, cmip6_hist_synth, cmip6_fut_synth = generate_synthetic_climate()

print(f"ERA5 synthétique     : {era5_synth.shape} — {era5_synth.time.values[0]} → {era5_synth.time.values[-1]}")
print(f"CMIP6 hist synthétique: {cmip6_hist_synth.shape}")
print(f"CMIP6 fut synthétique : {cmip6_fut_synth.shape}")
print(f"\nMoyenne ERA5    : {float(era5_synth.mean()):.1f}°C")
print(f"Moyenne CMIP6 hist: {float(cmip6_hist_synth.mean()):.1f}°C (biais = +{float(cmip6_hist_synth.mean()) - float(era5_synth.mean()):.1f}°C)")
print(f"Moyenne CMIP6 fut : {float(cmip6_fut_synth.mean()):.1f}°C")

In [ ]:
# --- Visualiser le biais GCM vs ERA5 ---
# Moyenne spatiale → série temporelle mensuelle
era5_monthly = era5_synth.mean(dim=["latitude", "longitude"]).resample(time="ME").mean()
cmip6h_monthly = cmip6_hist_synth.mean(dim=["latitude", "longitude"]).resample(time="ME").mean()

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Séries temporelles
axes[0].plot(era5_monthly.time, era5_monthly, label="ERA5 (obs)", color="#1565C0", alpha=0.8)
axes[0].plot(cmip6h_monthly.time, cmip6h_monthly, label="CMIP6 hist (brut)", color="#e63946", alpha=0.8)
axes[0].set_ylabel("Température (°C)")
axes[0].set_title("Température moyenne France — ERA5 vs CMIP6 historique (brut)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Biais mensuel
bias_monthly = cmip6h_monthly - era5_monthly
axes[1].bar(bias_monthly.time.values, bias_monthly.values, width=25, color="#e63946", alpha=0.6)
axes[1].axhline(y=0, color="black", linewidth=0.5)
axes[1].axhline(y=float(bias_monthly.mean()), color="#e63946", linestyle="--",
                label=f"Biais moyen = {float(bias_monthly.mean()):+.1f}°C")
axes[1].set_ylabel("Biais (°C)")
axes[1].set_title("Biais CMIP6 − ERA5")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Méthode 1 — Delta Method (la plus simple)

Le **delta method** est l'approche la plus simple et la plus utilisée en pratique :

$$T_{\text{projeté}}(t) = T_{\text{ERA5}}(t) + \Delta_{\text{GCM}}$$

où $\Delta_{\text{GCM}} = \overline{T}_{\text{GCM,futur}} - \overline{T}_{\text{GCM,hist}}$

**Avantages** : simple, préserve la variabilité observée, pas de changement de distribution
**Limites** : suppose que le biais est constant dans le temps, ne corrige pas les extrêmes

In [ ]:
def delta_method(era5, gcm_hist, gcm_fut, by_month=True):
    """Applique le delta method pour projeter ERA5 dans le futur.
    
    Parameters
    ----------
    era5 : xr.DataArray
        Observations historiques (ERA5)
    gcm_hist : xr.DataArray
        GCM historique (même période que ERA5)
    gcm_fut : xr.DataArray
        GCM futur (scénario SSP)
    by_month : bool
        Si True, calcule le delta par mois (capture la saisonnalité du changement)
    
    Returns
    -------
    projected : xr.DataArray
        ERA5 + delta GCM
    delta : xr.DataArray
        Le signal de changement climatique
    """
    if by_month:
        # Delta mensuel : capture le fait que le réchauffement est plus fort en été
        gcm_hist_clim = gcm_hist.groupby("time.month").mean()
        gcm_fut_clim = gcm_fut.groupby("time.month").mean()
        delta = gcm_fut_clim - gcm_hist_clim
        projected = era5.groupby("time.month") + delta
    else:
        # Delta global (scalaire)
        delta = gcm_fut.mean() - gcm_hist.mean()
        projected = era5 + delta
    
    projected.attrs = era5.attrs.copy()
    projected.attrs["method"] = "delta_method"
    
    return projected, delta


# Appliquer le delta method
era5_projected_delta, delta_signal = delta_method(
    era5_synth, cmip6_hist_synth, cmip6_fut_synth, by_month=True
)

# Visualiser le delta mensuel (un point central)
ilat, ilon = 5, 7  # ~46°N, 1.5°E (centre France)
delta_at_point = delta_signal.isel(latitude=ilat, longitude=ilon)

fig, ax = plt.subplots(figsize=(8, 4))
months = range(1, 13)
month_labels = ["Jan", "Fév", "Mar", "Avr", "Mai", "Jun",
                "Jul", "Aoû", "Sep", "Oct", "Nov", "Déc"]
colors = plt.cm.RdYlBu_r(np.linspace(0.2, 0.8, 12))
ax.bar(months, delta_at_point.values, color=colors, edgecolor="white", linewidth=0.5)
ax.set_xticks(months)
ax.set_xticklabels(month_labels)
ax.set_ylabel("ΔT (°C)")
ax.set_title(f"Signal de changement climatique (delta method) — Centre France\n"
             f"Moyenne annuelle : {float(delta_at_point.mean()):+.1f}°C")
ax.grid(True, alpha=0.3, axis="y")
ax.axhline(y=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## 6. Méthode 2 — Quantile Mapping (bias correction)

Le **quantile mapping** corrige la distribution complète du GCM pour la faire correspondre à celle d'ERA5 :

Pour chaque quantile $q$ :
$$T_{\text{corrigé}}(t) = F_{\text{ERA5}}^{-1}\left(F_{\text{GCM,hist}}\left(T_{\text{GCM}}(t)\right)\right)$$

En pratique :
1. On calcule la CDF empirique du GCM historique et d'ERA5
2. On mappe chaque valeur GCM vers le quantile ERA5 correspondant
3. On applique la même transformation aux données futures

**Avantages** : corrige tout le spectre (moyenne, variance, extrêmes)
**Limites** : suppose la stationnarité de la correction, peut dégrader le signal de tendance

In [ ]:
def quantile_mapping(obs, gcm_hist, gcm_target, n_quantiles=100, by_month=True):
    """Applique le quantile mapping pour corriger le biais d'un GCM.
    
    Parameters
    ----------
    obs : xr.DataArray
        Observations (ERA5)
    gcm_hist : xr.DataArray
        GCM historique (calibration)
    gcm_target : xr.DataArray
        GCM à corriger (historique ou futur)
    n_quantiles : int
        Nombre de quantiles pour la CDF empirique
    by_month : bool
        Si True, applique le QM par mois (meilleure capture saisonnière)
    
    Returns
    -------
    corrected : xr.DataArray
        GCM corrigé
    """
    quantiles = np.linspace(0, 1, n_quantiles + 1)
    
    def _qm_1d(obs_vals, hist_vals, target_vals):
        """Quantile mapping sur des vecteurs 1D."""
        obs_q = np.nanquantile(obs_vals, quantiles)
        hist_q = np.nanquantile(hist_vals, quantiles)
        # Interpolation : GCM value → quantile GCM → valeur ERA5 correspondante
        target_quantiles = np.interp(target_vals, hist_q, quantiles, left=0, right=1)
        corrected = np.interp(target_quantiles, quantiles, obs_q)
        return corrected
    
    if by_month:
        corrected_chunks = []
        for month in range(1, 13):
            obs_m = obs.sel(time=obs.time.dt.month == month)
            hist_m = gcm_hist.sel(time=gcm_hist.time.dt.month == month)
            target_m = gcm_target.sel(time=gcm_target.time.dt.month == month)
            
            corrected_m = xr.apply_ufunc(
                _qm_1d,
                obs_m.stack(space=("latitude", "longitude")),
                hist_m.stack(space=("latitude", "longitude")),
                target_m.stack(space=("latitude", "longitude")),
                input_core_dims=[["time"], ["time"], ["time"]],
                output_core_dims=[["time"]],
                vectorize=True,
            ).unstack("space")
            corrected_m["time"] = target_m.time
            corrected_chunks.append(corrected_m)
        
        corrected = xr.concat(corrected_chunks, dim="time").sortby("time")
    else:
        corrected = xr.apply_ufunc(
            _qm_1d,
            obs.stack(space=("latitude", "longitude")),
            gcm_hist.stack(space=("latitude", "longitude")),
            gcm_target.stack(space=("latitude", "longitude")),
            input_core_dims=[["time"], ["time"], ["time"]],
            output_core_dims=[["time"]],
            vectorize=True,
        ).unstack("space")
        corrected["time"] = gcm_target.time
    
    corrected.attrs = obs.attrs.copy()
    corrected.attrs["method"] = "quantile_mapping"
    return corrected


# Corriger le GCM historique (validation)
cmip6_hist_corrected = quantile_mapping(era5_synth, cmip6_hist_synth, cmip6_hist_synth)

# Corriger le GCM futur
cmip6_fut_corrected = quantile_mapping(era5_synth, cmip6_hist_synth, cmip6_fut_synth)

print(f"Biais avant correction : {float(cmip6_hist_synth.mean()) - float(era5_synth.mean()):+.2f}°C")
print(f"Biais après correction : {float(cmip6_hist_corrected.mean()) - float(era5_synth.mean()):+.2f}°C")

In [ ]:
# --- Validation du quantile mapping : comparaison des distributions ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Point central France
ilat, ilon = 5, 7

# QQ-plot : GCM brut vs ERA5
obs_vals = np.sort(era5_synth.isel(latitude=ilat, longitude=ilon).values)
gcm_vals = np.sort(cmip6_hist_synth.isel(latitude=ilat, longitude=ilon).values)
n = min(len(obs_vals), len(gcm_vals))
axes[0].scatter(obs_vals[::max(1, n//200)], gcm_vals[::max(1, n//200)],
                alpha=0.3, s=10, color="#e63946")
axes[0].plot([-15, 35], [-15, 35], "k--", linewidth=0.5)
axes[0].set_xlabel("ERA5 (°C)")
axes[0].set_ylabel("CMIP6 hist brut (°C)")
axes[0].set_title("QQ-plot : AVANT correction")
axes[0].set_aspect("equal")
axes[0].grid(True, alpha=0.3)

# QQ-plot : GCM corrigé vs ERA5
gcm_corr_vals = np.sort(cmip6_hist_corrected.isel(latitude=ilat, longitude=ilon).values)
axes[1].scatter(obs_vals[::max(1, n//200)], gcm_corr_vals[::max(1, n//200)],
                alpha=0.3, s=10, color="#2196F3")
axes[1].plot([-15, 35], [-15, 35], "k--", linewidth=0.5)
axes[1].set_xlabel("ERA5 (°C)")
axes[1].set_ylabel("CMIP6 hist corrigé (°C)")
axes[1].set_title("QQ-plot : APRÈS correction")
axes[1].set_aspect("equal")
axes[1].grid(True, alpha=0.3)

# Histogrammes comparés
bins = np.arange(-15, 35, 1)
axes[2].hist(era5_synth.isel(latitude=ilat, longitude=ilon).values, bins=bins,
             alpha=0.5, density=True, label="ERA5", color="#1565C0")
axes[2].hist(cmip6_hist_synth.isel(latitude=ilat, longitude=ilon).values, bins=bins,
             alpha=0.3, density=True, label="CMIP6 brut", color="#e63946")
axes[2].hist(cmip6_hist_corrected.isel(latitude=ilat, longitude=ilon).values, bins=bins,
             alpha=0.3, density=True, label="CMIP6 corrigé", color="#4CAF50")
axes[2].set_xlabel("Température (°C)")
axes[2].set_ylabel("Densité")
axes[2].set_title("Distribution — Centre France")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Calcul des indicateurs climatiques

On définit une bibliothèque d'indicateurs calculables à partir de données journalières de température et précipitations. Chaque indicateur est calculé sur la baseline ERA5 puis sur les projections corrigées.

In [ ]:
# ── Bibliothèque d'indicateurs climatiques ────────────────────────────────────

def hot_days(tas, threshold=35.0):
    """Nombre de jours par an où T > seuil (°C)."""
    return (tas > threshold).groupby("time.year").sum(dim="time")

def tropical_nights(tasmin, threshold=20.0):
    """Nombre de nuits par an où Tmin > seuil (°C).
    Note: ici on utilise tas comme proxy de tasmin."""
    return (tasmin > threshold).groupby("time.year").sum(dim="time")

def frost_days(tasmin, threshold=0.0):
    """Nombre de jours par an où Tmin < 0°C."""
    return (tasmin < threshold).groupby("time.year").sum(dim="time")

def cooling_degree_days(tas, base=22.0):
    """Degrés-jours de refroidissement (base 22°C) par an.
    CDD = Σ max(0, T - base)"""
    return tas.where(tas > base, 0).groupby("time.year").apply(
        lambda x: (x - base).clip(min=0).sum(dim="time")
    )

def heating_degree_days(tas, base=18.0):
    """Degrés-jours de chauffage (base 18°C) par an.
    HDD = Σ max(0, base - T)"""
    return tas.groupby("time.year").apply(
        lambda x: (base - x).clip(min=0).sum(dim="time")
    )

def mean_annual_temp(tas):
    """Température moyenne annuelle."""
    return tas.groupby("time.year").mean(dim="time")

def summer_mean_temp(tas):
    """Température moyenne estivale (JJA)."""
    summer = tas.sel(time=tas.time.dt.month.isin([6, 7, 8]))
    return summer.groupby("time.year").mean(dim="time")

# ── Calculer les indicateurs ──────────────────────────────────────────────────
indicators = {}

# Sur ERA5 (baseline)
indicators["hot_days_baseline"] = hot_days(era5_synth)
indicators["frost_days_baseline"] = frost_days(era5_synth)
indicators["cdd_baseline"] = cooling_degree_days(era5_synth)
indicators["hdd_baseline"] = heating_degree_days(era5_synth)
indicators["mean_temp_baseline"] = mean_annual_temp(era5_synth)

# Sur projections (delta method)
indicators["hot_days_delta"] = hot_days(era5_projected_delta)
indicators["frost_days_delta"] = frost_days(era5_projected_delta)
indicators["cdd_delta"] = cooling_degree_days(era5_projected_delta)
indicators["hdd_delta"] = heating_degree_days(era5_projected_delta)
indicators["mean_temp_delta"] = mean_annual_temp(era5_projected_delta)

# Sur projections (quantile mapping)
indicators["hot_days_qm"] = hot_days(cmip6_fut_corrected)
indicators["frost_days_qm"] = frost_days(cmip6_fut_corrected)
indicators["cdd_qm"] = cooling_degree_days(cmip6_fut_corrected)
indicators["hdd_qm"] = heating_degree_days(cmip6_fut_corrected)
indicators["mean_temp_qm"] = mean_annual_temp(cmip6_fut_corrected)

print("Indicateurs calculés :")
for name in ["hot_days", "frost_days", "cdd", "hdd", "mean_temp"]:
    base_val = float(indicators[f"{name}_baseline"].mean())
    delta_val = float(indicators[f"{name}_delta"].mean())
    qm_val = float(indicators[f"{name}_qm"].mean())
    print(f"  {name:20s} | baseline: {base_val:8.1f} | delta: {delta_val:8.1f} | QM: {qm_val:8.1f}")

In [ ]:
# --- Visualisation des indicateurs : baseline vs projections ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ilat, ilon = 5, 7  # Centre France

# 1. Jours de chaleur extrême
ax = axes[0, 0]
years_b = indicators["hot_days_baseline"].year.values
years_d = indicators["hot_days_delta"].year.values
ax.bar(years_b, indicators["hot_days_baseline"].isel(latitude=ilat, longitude=ilon).values,
       color="#1565C0", alpha=0.7, label="Baseline ERA5")
ax.bar(years_d, indicators["hot_days_delta"].isel(latitude=ilat, longitude=ilon).values,
       color="#e63946", alpha=0.7, label="Projeté (delta)")
ax.set_ylabel("Jours/an")
ax.set_title("Jours de chaleur extrême (T > 35°C)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 2. Jours de gel
ax = axes[0, 1]
ax.bar(years_b, indicators["frost_days_baseline"].isel(latitude=ilat, longitude=ilon).values,
       color="#1565C0", alpha=0.7, label="Baseline ERA5")
ax.bar(years_d, indicators["frost_days_delta"].isel(latitude=ilat, longitude=ilon).values,
       color="#2196F3", alpha=0.7, label="Projeté (delta)")
ax.set_ylabel("Jours/an")
ax.set_title("Jours de gel (T < 0°C)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 3. CDD (refroidissement)
ax = axes[1, 0]
ax.bar(years_b, indicators["cdd_baseline"].isel(latitude=ilat, longitude=ilon).values,
       color="#1565C0", alpha=0.7, label="Baseline ERA5")
ax.bar(years_d, indicators["cdd_delta"].isel(latitude=ilat, longitude=ilon).values,
       color="#FF9800", alpha=0.7, label="Projeté (delta)")
ax.set_ylabel("°C·jours/an")
ax.set_title("Degrés-jours de refroidissement (base 22°C)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# 4. HDD (chauffage)
ax = axes[1, 1]
ax.bar(years_b, indicators["hdd_baseline"].isel(latitude=ilat, longitude=ilon).values,
       color="#1565C0", alpha=0.7, label="Baseline ERA5")
ax.bar(years_d, indicators["hdd_delta"].isel(latitude=ilat, longitude=ilon).values,
       color="#9C27B0", alpha=0.7, label="Projeté (delta)")
ax.set_ylabel("°C·jours/an")
ax.set_title("Degrés-jours de chauffage (base 18°C)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle("Indicateurs climatiques — Centre France (~46°N, 1.5°E)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 8. Cartes spatiales des indicateurs

Visualisation spatiale du changement climatique sur la France : comparaison baseline vs projections.

In [ ]:
# --- Cartes : changement climatique spatial ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10),
                          subplot_kw={"projection": None})

def _plot_map(ax, data, title, cmap, vmin, vmax, label):
    im = ax.pcolormesh(
        data.longitude, data.latitude, data.values,
        cmap=cmap, vmin=vmin, vmax=vmax, shading="auto",
    )
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, label=label, shrink=0.8)

# Moyenne climatologique (30 ans)
mean_temp_base = indicators["mean_temp_baseline"].mean(dim="year")
mean_temp_proj = indicators["mean_temp_delta"].mean(dim="year")
delta_temp = mean_temp_proj - mean_temp_base

hot_days_base = indicators["hot_days_baseline"].mean(dim="year")
hot_days_proj = indicators["hot_days_delta"].mean(dim="year")
delta_hot = hot_days_proj - hot_days_base

# Ligne 1 : Température moyenne
_plot_map(axes[0, 0], mean_temp_base,
          "T moyenne — Baseline (1991–2020)", "RdYlBu_r", 6, 18, "°C")
_plot_map(axes[0, 1], mean_temp_proj,
          "T moyenne — Projeté (2041–2070)", "RdYlBu_r", 6, 18, "°C")
_plot_map(axes[0, 2], delta_temp,
          "ΔT moyenne", "Reds", 0, 4, "°C")

# Ligne 2 : Jours de chaleur
_plot_map(axes[1, 0], hot_days_base.astype(float),
          "Jours > 35°C — Baseline", "YlOrRd", 0, 30, "jours/an")
_plot_map(axes[1, 1], hot_days_proj.astype(float),
          "Jours > 35°C — Projeté", "YlOrRd", 0, 30, "jours/an")
_plot_map(axes[1, 2], delta_hot.astype(float),
          "Δ Jours > 35°C", "Reds", 0, 20, "jours/an")

plt.suptitle("Changement climatique spatial — France\n"
             "Delta method, CMIP6 SSP (warming = +2.5°C)", fontsize=13)
plt.tight_layout()
plt.show()

## 9. Comparaison des méthodes de downscaling

Récapitulatif des différences entre delta method et quantile mapping sur nos indicateurs.

In [ ]:
# --- Tableau comparatif des méthodes ---
ilat, ilon = 5, 7  # Centre France

comparison_data = []
for name, label, unit in [
    ("hot_days", "Jours > 35°C", "j/an"),
    ("frost_days", "Jours de gel", "j/an"),
    ("cdd", "CDD (base 22°C)", "°C·j/an"),
    ("hdd", "HDD (base 18°C)", "°C·j/an"),
    ("mean_temp", "T moyenne", "°C"),
]:
    base = float(indicators[f"{name}_baseline"].isel(latitude=ilat, longitude=ilon).mean())
    delta = float(indicators[f"{name}_delta"].isel(latitude=ilat, longitude=ilon).mean())
    qm = float(indicators[f"{name}_qm"].isel(latitude=ilat, longitude=ilon).mean())
    comparison_data.append({
        "Indicateur": label,
        "Unité": unit,
        "Baseline": f"{base:.1f}",
        "Delta method": f"{delta:.1f}",
        "Quantile mapping": f"{qm:.1f}",
        "Δ (delta)": f"{delta - base:+.1f}",
        "Δ (QM)": f"{qm - base:+.1f}",
    })

df_comparison = pd.DataFrame(comparison_data)
print("Comparaison des méthodes — Centre France (~46°N, 1.5°E)")
print("=" * 90)
print(df_comparison.to_string(index=False))
print("\nNotes :")
print("- Delta method préserve la variabilité observée, QM corrige la distribution")
print("- Les deux convergent sur la moyenne mais divergent sur les extrêmes")
print("- Pour les indicateurs de seuil (hot days, frost days), QM est préférable")

## 10. Regridding avec xESMF (CMIP6 → ERA5)

Quand on travaille avec des données réelles, les GCMs CMIP6 ont des grilles différentes (1-2°) de ERA5 (0.25°). Le **regridding** est nécessaire pour les mettre sur la même grille avant la bias correction.

`xesmf` offre plusieurs méthodes :
- **bilinear** : interpolation bilinéaire (rapide, bon pour T)
- **conservative** : conserve les flux (nécessaire pour précipitations)
- **nearest_s2d** : plus proche voisin (pas d'artefacts de lissage)

In [ ]:
import xesmf as xe

# Démonstration : regrider un champ CMIP6 (1°) vers une grille fine (0.25°)
# Grille source (CMIP6 ~1°)
ds_coarse = cmip6_hist_synth.isel(time=0).to_dataset(name="tas")

# Grille cible (ERA5 0.25°)
lat_fine = np.arange(41.0, 51.5, 0.25)
lon_fine = np.arange(-5.5, 10.0, 0.25)
ds_fine = xr.Dataset({
    "latitude": (["latitude"], lat_fine),
    "longitude": (["longitude"], lon_fine),
})

# Créer le regridder
regridder = xe.Regridder(ds_coarse, ds_fine, method="bilinear")
print(regridder)

# Appliquer le regridding
tas_regridded = regridder(ds_coarse["tas"])

# Visualiser
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ds_coarse["tas"].plot(ax=axes[0], cmap="RdYlBu_r")
axes[0].set_title(f"CMIP6 original ({len(ds_coarse.latitude)}×{len(ds_coarse.longitude)} = 1°)")

tas_regridded.plot(ax=axes[1], cmap="RdYlBu_r")
axes[1].set_title(f"Regridded bilinéaire ({len(lat_fine)}×{len(lon_fine)} = 0.25°)")

plt.suptitle("Regridding CMIP6 → grille ERA5 avec xESMF")
plt.tight_layout()
plt.show()

## 11. Pipeline complet BCSD (Bias Correction + Spatial Disaggregation)

Le **BCSD** combine les deux approches précédentes :

1. **Regrid** le GCM vers la grille grossière d'ERA5 (ou une grille commune)
2. **Bias correct** par quantile mapping sur la grille grossière
3. **Spatial disaggregation** : ajoute le détail spatial d'ERA5 au signal corrigé

$$T_{\text{BCSD}}(x, t) = T_{\text{ERA5,clim}}(x) + \Delta T_{\text{BC}}(X, t)$$

où $x$ = haute résolution, $X$ = basse résolution, $\Delta T_{\text{BC}}$ = anomalie corrigée.

In [ ]:
def bcsd_pipeline(era5_fine, gcm_hist_coarse, gcm_fut_coarse, n_quantiles=100):
    """Pipeline BCSD simplifié.
    
    Étapes :
    1. Calcul des climatologies mensuelles ERA5 haute résolution
    2. Bias correction QM sur grille grossière (GCM)
    3. Calcul des anomalies corrigées
    4. Regridding des anomalies vers la grille fine
    5. Ajout de la climatologie ERA5 fine
    
    Parameters
    ----------
    era5_fine : xr.DataArray
        ERA5 à haute résolution (0.25°)
    gcm_hist_coarse : xr.DataArray
        GCM historique à basse résolution (~1°)
    gcm_fut_coarse : xr.DataArray
        GCM futur à basse résolution (~1°)
    
    Returns
    -------
    projected_fine : xr.DataArray
        Projections à haute résolution
    """
    # 1. Climatologie mensuelle ERA5 (haute résolution)
    era5_monthly_clim = era5_fine.groupby("time.month").mean()
    
    # 2. Bias correction QM sur grille grossière
    gcm_fut_corrected = quantile_mapping(
        # On doit d'abord agréger ERA5 sur la grille GCM pour la calibration
        era5_fine,  # Utilisation directe ici car même grille dans notre exemple
        gcm_hist_coarse,
        gcm_fut_coarse,
        n_quantiles=n_quantiles,
        by_month=True,
    )
    
    # 3. Anomalies corrigées (par rapport à la climatologie GCM corrigée)
    gcm_fut_monthly_clim = gcm_fut_corrected.groupby("time.month").mean()
    anomalies = gcm_fut_corrected.groupby("time.month") - gcm_fut_monthly_clim
    
    # 4-5. Reconstruction : climatologie ERA5 fine + anomalies
    # (Dans notre cas synthétique, même grille, donc pas de regridding nécessaire)
    projected_fine = anomalies.groupby("time.month") + era5_monthly_clim
    
    projected_fine.attrs = era5_fine.attrs.copy()
    projected_fine.attrs["method"] = "BCSD"
    
    return projected_fine


# Appliquer BCSD
cmip6_fut_bcsd = bcsd_pipeline(era5_synth, cmip6_hist_synth, cmip6_fut_synth)

# Comparer les 3 méthodes
print("Température moyenne projetée (Centre France) :")
print(f"  Delta method     : {float(era5_projected_delta.isel(latitude=5, longitude=7).mean()):.1f}°C")
print(f"  Quantile mapping : {float(cmip6_fut_corrected.isel(latitude=5, longitude=7).mean()):.1f}°C")
print(f"  BCSD             : {float(cmip6_fut_bcsd.isel(latitude=5, longitude=7).mean()):.1f}°C")

## 12. Récapitulatif et prochaines étapes

### Ce qu'on a construit dans ce notebook

| Étape | Outil | Statut |
|-------|-------|--------|
| Téléchargement ERA5 | `cdsapi` | Fonctions prêtes (décommenter pour lancer) |
| Téléchargement CMIP6 | `cdsapi` | Fonctions prêtes |
| Données synthétiques | NumPy/xarray | Opérationnel (démo du pipeline) |
| Delta method | Custom | Implémenté + validé |
| Quantile mapping | Custom | Implémenté + validé (QQ-plots) |
| BCSD pipeline | Custom + xesmf | Implémenté |
| Regridding | `xesmf` | Démontré (bilinéaire) |
| Indicateurs climatiques | 7 indicateurs | Calculés et visualisés |

### Recommandations par cas d'usage

| Cas d'usage | Méthode recommandée |
|-------------|---------------------|
| Screening rapide, tendance moyenne | **Delta method** |
| Indicateurs de seuil (jours > 35°C) | **Quantile mapping** |
| Analyse spatiale haute résolution | **BCSD** |
| Production opérationnelle | QM par mois + multi-GCM ensemble |

### Prochaines étapes
- [ ] Configurer le compte CDS et télécharger les données réelles
- [ ] Appliquer le pipeline sur les 5 modèles CMIP6 × 4 scénarios SSP
- [ ] Ajouter les précipitations (pr) et le vent (sfcWind)
- [ ] Intégrer les indicateurs dans `src/open_climate_risk/` comme module
- [ ] Connecter avec l'analyse de risque existante (EAD → indicateurs projetés)
- [ ] Explorer les datasets downscalés existants (CORDEX, NEX-GDDP-CMIP6) comme alternative